A hyper-parameter is a parameter whose value is chosen by the egnineer (its dynamic)
Multi-Layer Perceptron 

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open("names.txt", "r").read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)} #creating Lookup table -string to integer mapping, +1 added so special character can have index position 0
stoi['.'] = 0 #special character
itos = {i:s for s,i in stoi.items()} #integer to string mapping - inverse of stoi 
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset

block_size = 3 #context length: how many characters do we take to predict the next one?
X,Y = [], []

for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '----->', itos[ix])
        context = context[1:] + [ix] #crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... -----> e
..e -----> m
.em -----> m
emm -----> a
mma -----> .
olivia
... -----> o
..o -----> l
.ol -----> i
oli -----> v
liv -----> i
ivi -----> a
via -----> .
ava
... -----> a
..a -----> v
.av -----> a
ava -----> .
isabella
... -----> i
..i -----> s
.is -----> a
isa -----> b
sab -----> e
abe -----> l
bel -----> l
ell -----> a
lla -----> .
sophia
... -----> s
..s -----> o
.so -----> p
sop -----> h
oph -----> i
phi -----> a
hia -----> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
# build the dataset to get training, validation and test split

def build_dataset(words):
    block_size = 3 #context length: how many characters do we take to predict the next one
    X, Y = [], []
    for w in words:

        # print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '----->', itos[ix])
            context = context[1:] + [ix] #crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1]) #get test split
Xdev, Ydev = build_dataset(words[n1:n2]) # get dev/validation split
Xte, Yte = build_dataset(words[n1:]) #get test split

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([45521, 3]) torch.Size([45521])


In [8]:
len(words)

32033

In [9]:
n1

25626

In [10]:
n2

28829

In [11]:
C = torch.randn((27,2))

In [12]:
emb = C[X]   #embed all of the integers of X into C to create lookup table
emb.shape

torch.Size([32, 3, 2])

In [13]:
w1 = torch.randn((6,100))
b1 = torch.randn(100)

In [14]:
h = torch.tanh(emb.view(-1,6) @ w1 + b1) 
# .view is to transform the matrix from a [32, 3, 2] to [32, 6] so we can perfrom multiplication
# tanh is to perfrom softmax and make the negative numbers positive


In [15]:
h.shape

torch.Size([32, 100])

In [16]:
w2 =torch.randn((100,27))
b2 =torch.randn(27)

In [17]:
logits = h @ w2 + b2

In [18]:
logits.shape

torch.Size([32, 27])

In [19]:
counts = logits.exp()

In [20]:
prob = counts / counts.sum(1, keepdims=True)

In [21]:
prob.shape

torch.Size([32, 27])

In [22]:
loss = -prob[torch.arange(32), Y].log().mean()    # negative log likelihood loss
loss

tensor(13.8636)

In [23]:
# ----------------now the whole code so far arranged ---------------------

In [24]:
Xtr.shape, Ytr.shape

(torch.Size([182625, 3]), torch.Size([182625]))

In [ ]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,10), generator=g) #embedding table, 27 characters and 10 dimensional embedding vector for each character
# increasing the w1 from 6x100 to 6x100 meanigs increasing the size of the hidden layer and the neural network
w1 = torch.randn((30,200), generator=g)
b1 = torch.randn(200, generator=g) # 
w2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, b1, w2, b2]

In [26]:
sum(p.nelement() for p in parameters ) # number of parameters in total

11897

In [27]:
for p in parameters:
    p.requires_grad=True

In [28]:
# to get a learning rate that would be good to use
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

counts  = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
The above 3 lines is known as classifications and can be done directly by using the pytorch function  cross_entropy

In [29]:
lri = []
lossi = []
stepi = []

In [30]:
for i in range(200000):

    # minibatch construct - to perform the gradient descent in 32-minibatches of the data to make it more faster
    ix = torch.randint(0, Xtr.shape[0], (32,))

    #Forward pass
    emb = C[Xtr[ix]] #[32, 3, 2]
    h =  torch.tanh(emb.view(-1, 30) @ w1 + b1)  #(32, 100)
    logits = h @  w2 + b2 #(32, 27)
    loss  =  F.cross_entropy(logits, Ytr[ix])

    # backward pass
    for p in parameters:
        p.grad = None   #same as setting to 0
    loss.backward()     #to populate the gradients

    # update
    # lr = lrs[i]
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    # lri.append(lre[i])
    stepi.append(i)
    lossi.append(loss.log10().item())
        
print(loss.item())

2.342428684234619


learning rate decay is the processing of reducing the learnign rate (from 0.1  to 0.01) and train the model for a few more steps

In [31]:
# a  graph where the bottommm shows the number with a good learning rate
# plt.plot(lri, lossi)
# plt.plot(stepi, lossi)

In [32]:
# To avoid a sitaution where the model memorizes our dataset (known as overfitting), we split the data into 3 categories
#training split, dev/validation split, test split
# 80% of dataset, 10%, 10%

# training split 80% - used to optimize the parameters of the model (forward pass, BP and update - AKA gradient descent)
# dev/validation split 10% = used for development of the hyperparameter E.g the size of the hidden layer, size of the embedding
# test split 10% = used to evaluate the performance of the model at the end - should not be done often

In [33]:
# to evaluate the loss  using the dev split

emb = C[Xdev] #[32, 3, 2]
h =  torch.tanh(emb.view(-1, 30) @ w1 + b1)  #(32, 100)
logits = h @  w2 + b2 #(32, 27)
loss  =  F.cross_entropy(logits, Ydev)
loss

tensor(2.1795, grad_fn=<NllLossBackward0>)

In [ ]:
# to evaluate the loss  using the test split  (should not be done often)

emb = C[Xte] #[32, 3, 2]
h =  torch.tanh(emb.view(-1, 30) @ w1 + b1)  #(32, 100)
logits = h @  w2 + b2 #(32, 27)
loss  =  F.cross_entropy(logits, Yte)
loss

tensor(2.1776, grad_fn=<NllLossBackward0>)

In [39]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size #initialize with all ...
    while True:
        emb = C[torch.tensor([context])] # (1,block_size,d)
        h = torch.tanh(emb.view(1, -1) @ w1 + b1)
        logits = h @ w2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context  = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))
        

mora.
kayah.
seer.
nihah.
lorethruchadrie.
caileydielin.
shi.
jen.
eden.
sananar.
kayzion.
kamin.
shubprishiriel.
kindreelynn.
nophir.
ure.
ged.
ryyah.
fael.
yuma.
